In [ ]:
import cv2
import numpy as np


# ---------------------------------------------------------
# 1. Freeman Chain Code
# ---------------------------------------------------------

# Mapping from movement (dx, dy) to Freeman direction
direction_code = {
    (1, 0): 0,      # Right
    (1, -1): 1,     # Up-right
    (0, -1): 2,     # Up
    (-1, -1): 3,    # Up-left
    (-1, 0): 4,     # Left
    (-1, 1): 5,     # Down-left
    (0, 1): 6,      # Down
    (1, 1): 7       # Down-right
}


def get_chain_code(contour):
    """
    Generate Freeman 8-direction chain code
    from an OpenCV contour.
    """

    # Convert contour from shape (N,1,2) to (N,2)
    points = contour.reshape(-1, 2)

    chain_code = []

    for i in range(len(points)):

        # Current point
        x1, y1 = points[i]

        # Next point
        x2, y2 = points[(i + 1) % len(points)]

        dx = x2 - x1
        dy = y2 - y1

        # Normalize movement to -1, 0 or 1
        dx = int(np.sign(dx))
        dy = int(np.sign(dy))

        if (dx, dy) in direction_code:
            chain_code.append(direction_code[(dx, dy)])

    return chain_code


# ---------------------------------------------------------
# 2. First Difference
# ---------------------------------------------------------

def first_difference(chain_code):
    """
    Compute first difference of the chain code.

    Difference = (next_direction - current_direction) mod 8
    """

    diff = []

    n = len(chain_code)

    for i in range(n):

        current = chain_code[i]
        next_code = chain_code[(i + 1) % n]

        d = (next_code - current) % 8

        diff.append(d)

    return diff


# ---------------------------------------------------------
# 3. Shape Number
# ---------------------------------------------------------

def get_shape_number(chain_code):
    """
    Shape number = lexicographically smallest
    circular shift of the first difference.
    """

    diff = first_difference(chain_code)

    if len(diff) == 0:
        return []

    rotations = []

    for i in range(len(diff)):
        rotation = diff[i:] + diff[:i]
        rotations.append(rotation)

    # Smallest circular sequence
    shape_number = min(rotations)

    return shape_number


# ---------------------------------------------------------
# 4. Load image
# ---------------------------------------------------------

image = cv2.imread("assets/Shape2.png")

if image is None:
    raise FileNotFoundError("Image could not be loaded.")

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


# ---------------------------------------------------------
# 5. Convert image to binary
# ---------------------------------------------------------

_, binary = cv2.threshold(
    gray,
    127,
    255,
    cv2.THRESH_BINARY_INV
)


# If the object is black and background white,
# use THRESH_BINARY_INV instead:
#
# _, binary = cv2.threshold(
#   gray, 127, 255, cv2.THRESH_BINARY_INV
#)

# If the object is white and background black,
# use THRESH_BINARY instead:
#
# _, binary = cv2.threshold(
#   gray, 127, 255, cv2.THRESH_BINARY
#)

# ---------------------------------------------------------
# 6. Extract contours
# ---------------------------------------------------------

contours, hierarchy = cv2.findContours(
    binary,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_NONE
)


# Check that a contour exists
if len(contours) == 0:
    raise ValueError("No contour detected.")


# Select the largest object
contour = max(contours, key=cv2.contourArea)


# ---------------------------------------------------------
# 7. Obtain Chain Code
# ---------------------------------------------------------

chain_code = get_chain_code(contour)

print("Freeman Chain Code:")
print(chain_code)


# ---------------------------------------------------------
# 8. First Difference
# ---------------------------------------------------------

difference = first_difference(chain_code)

print("\nFirst Difference:")
print(difference)


# ---------------------------------------------------------
# 9. Shape Number
# ---------------------------------------------------------

shape_number = get_shape_number(chain_code)

print("\nShape Number:")
print(shape_number)


# ---------------------------------------------------------
# 10. Draw contour
# ---------------------------------------------------------

output = image.copy()

cv2.drawContours(
    output,
    [contour],
    -1,
    (0, 0, 255),
    2
)

cv2.imshow("Original Image", image)
cv2.imshow("Binary Image", binary)
cv2.imshow("Detected Contour", output)

cv2.waitKey(0)
cv2.destroyAllWindows()

Freeman Chain Code:
[5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 6, 5, 5, 6, 5, 6, 5, 6, 6, 5, 6, 5, 5, 6, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 6, 5, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 7, 7, 7, 6, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 6, 7, 0, 7, 6, 7, 0, 7, 6, 0, 7, 7, 7, 7